# DietNitrateOralEcologyPaperVisualizer

Use this notebook outside TRE after copying the `result_export` folder. It reads de-identified summary/statistics CSVs exported by `DietNitrateOralEcologyPaper.ipynb` and creates editable publication-style plots. No participant IDs are required.


In [ ]:
from pathlib import Path
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RESULT_EXPORT_DIR = Path("./result_export")
ANALYSIS_PREFIX = "A_main"  # Change to "B_secondary_logging_covariates" for sensitivity analysis.

BUNDLE_FILES = {
    "A_main": "A_main_result_export_bundle.csv",
    "B_secondary_logging_covariates": "B_secondary_result_export_bundle.csv",
}

_bundle_cache = {}


def parse_plain_csv_bundle(bundle_path):
    bundle_path = Path(bundle_path)
    tables = {}
    current_name = None
    current_lines = []

    with open(bundle_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            if line.startswith("#") or line == "":
                continue

            if line.startswith("__TABLE_START__,"):
                parts = line.split(",", 4)
                current_name = parts[1]
                current_lines = []
                continue

            if line.startswith("__TABLE_END__,"):
                if current_name is not None:
                    csv_text = "\n".join(current_lines)
                    tables[current_name] = pd.read_csv(StringIO(csv_text)) if csv_text.strip() else pd.DataFrame()
                current_name = None
                current_lines = []
                continue

            if current_name is not None:
                current_lines.append(line)

    return tables


def get_bundle_tables(prefix=ANALYSIS_PREFIX):
    if prefix not in _bundle_cache:
        bundle_name = BUNDLE_FILES.get(prefix, f"{prefix}_result_export_bundle.csv")
        bundle_path = RESULT_EXPORT_DIR / bundle_name
        if not bundle_path.exists():
            raise FileNotFoundError(
                f"Could not find individual CSVs or bundle for prefix {prefix}. Expected bundle: {bundle_path}"
            )
        _bundle_cache[prefix] = parse_plain_csv_bundle(bundle_path)
    return _bundle_cache[prefix]


def read_export(name, prefix=ANALYSIS_PREFIX):
    # Preferred when many individual CSVs are available.
    path = RESULT_EXPORT_DIR / f"{prefix}_{name}.csv"
    if path.exists():
        return pd.read_csv(path)

    # TRE export fallback: one plain CSV bundle containing many tables.
    tables = get_bundle_tables(prefix)
    if name not in tables:
        available = ", ".join(sorted(tables.keys())[:30])
        raise KeyError(f"Table {name!r} not found in bundle for {prefix}. Available examples: {available}")
    return tables[name].copy()


def list_available_exports(prefix=ANALYSIS_PREFIX):
    individual = sorted([p.name.replace(f"{prefix}_", "").replace(".csv", "") for p in RESULT_EXPORT_DIR.glob(f"{prefix}_*.csv")])
    if individual:
        return individual
    return sorted(get_bundle_tables(prefix).keys())


def pretty_label(x):
    return str(x).replace("_", " ").replace("oral metaphlan ", "oral ")

print("Using analysis prefix:", ANALYSIS_PREFIX)
print("Available exports:", list_available_exports()[:40])


## Load Exported Tables


In [ ]:
main_ecology = read_export("main_ecology_results")
directional_taxa = read_export("directional_taxa_lists")
opposite_axis = read_export("opposite_axis_candidates")
plot_group_summary = read_export("plot_group_distribution_summary")
stratified_demo_summary = read_export("stratified_demographic_plot_distribution_summary")
manifest = read_export("export_manifest")

print("Loaded tables from individual CSV files or bundle:")
print("main_ecology", main_ecology.shape)
print("directional_taxa", directional_taxa.shape)
print("opposite_axis", opposite_axis.shape)
print("plot_group_summary", plot_group_summary.shape)
print("stratified_demo_summary", stratified_demo_summary.shape)

display(manifest)
display(main_ecology.head())
display(opposite_axis.head())


## Cross-Axis Effect Heatmap


In [ ]:
def plot_effect_heatmap(main_ecology, outcomes=None, exposures=None):
    df = main_ecology.copy()
    if outcomes is None:
        outcomes = df.sort_values(["exposure_q_value", "exposure_p"], na_position="last")["outcome"].drop_duplicates().head(25).tolist()
    if exposures is None:
        exposures = df["exposure"].drop_duplicates().tolist()
    mat = df.pivot_table(index="outcome", columns="exposure", values="exposure_standardized_beta", aggfunc="first").reindex(index=outcomes, columns=exposures)
    qmat = df.pivot_table(index="outcome", columns="exposure", values="exposure_q_value", aggfunc="first").reindex(index=outcomes, columns=exposures)
    vmax = np.nanmax(np.abs(mat.to_numpy(dtype=float)))
    vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1
    fig, ax = plt.subplots(figsize=(11, max(5, 0.38 * len(outcomes))), dpi=140)
    im = ax.imshow(mat, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(np.arange(len(exposures)))
    ax.set_xticklabels([pretty_label(x) for x in exposures], rotation=35, ha="right")
    ax.set_yticks(np.arange(len(outcomes)))
    ax.set_yticklabels([pretty_label(x) for x in outcomes], fontsize=8)
    for i in range(len(outcomes)):
        for j in range(len(exposures)):
            q = qmat.iloc[i, j]
            if np.isfinite(q):
                ax.text(j, i, f"q={q:.2g}", ha="center", va="center", fontsize=7)
    fig.colorbar(im, ax=ax, label="Adjusted standardized beta")
    ax.set_title(f"{ANALYSIS_PREFIX}: dietary nitrogen axes and oral ecology")
    fig.tight_layout()
    return fig, ax

fig, ax = plot_effect_heatmap(main_ecology)
plt.show()


## Same Bacterium Across Axes


In [ ]:
def plot_outcome_across_axes(outcome):
    sub = main_ecology[main_ecology["outcome"] == outcome].copy()
    sub = sub.sort_values("exposure")
    fig, ax = plt.subplots(figsize=(9, 4.8), dpi=140)
    colors = np.where(sub["exposure_standardized_beta"] >= 0, "#b2182b", "#2166ac")
    ax.bar(np.arange(len(sub)), sub["exposure_standardized_beta"], color=colors, edgecolor="black", alpha=0.85)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xticks(np.arange(len(sub)))
    ax.set_xticklabels([pretty_label(x) for x in sub["exposure"]], rotation=30, ha="right")
    for i, (_, row) in enumerate(sub.iterrows()):
        beta = row["exposure_standardized_beta"]
        ax.text(i, beta, f"p={row['exposure_p']:.2g}\nq={row['exposure_q_value']:.2g}\nd={row['high_minus_low_d']:.2g}", ha="center", va="bottom" if beta >= 0 else "top", fontsize=7)
    ax.set_ylabel("Adjusted standardized beta")
    ax.set_title(pretty_label(outcome))
    ax.grid(True, axis="y", color="black", alpha=0.2)
    fig.tight_layout()
    return fig, ax

for outcome in opposite_axis["outcome"].head(4):
    fig, ax = plot_outcome_across_axes(outcome)
    plt.show()


## Exported Group Summary Tables


In [ ]:
display(plot_group_summary.head(30))
display(stratified_demo_summary.head(30))
